---
# Personalised Recommendation Associaton Rules
---

## Library Imports

In [1]:
# ============================================================================
# LIBRARY IMPORTS
# ============================================================================
import pandas as pd
from prefixspan import PrefixSpan

# To suppress warnings during execution
import warnings

In [2]:
# Suppress all warnings while executing the code to keep output clean
warnings.filterwarnings('ignore')

---
# Association Rules | Prefix Span Model
---

In [3]:

# Load data
borrowings = pd.read_csv("../../../data/processed/library_borrowings.csv")
borrowings['borrowing date'] = pd.to_datetime(borrowings['borrowing date'])

# Sort by reader and borrowing date
borrowings_sorted = borrowings.sort_values(['Reader_num', 'borrowing date'])

# Create sequences of book titles for each reader
sequences = []
for reader_num, group in borrowings_sorted.groupby('Reader_num'):
    title_sequence = group['Title'].tolist()
    if len(title_sequence) > 1:
        sequences.append(title_sequence)

total_sequences = len(sequences)

# Run PrefixSpan
min_support = 1
ps = PrefixSpan(sequences)
patterns = ps.frequent(min_support)

# Calculate confidence for sequential rules
def calculate_confidence(patterns, total_sequences):
    """
    For a pattern [A, B], confidence = support(A → B) / support(A)
    Support = count / total_sequences
    """
    rules = []
    
    # Get support for single items (antecedents)
    single_item_support = {}
    for count, pattern in patterns:
        if len(pattern) == 1:
            single_item_support[pattern[0]] = count
    
    # Calculate confidence for patterns of length >= 2
    for count, pattern in patterns:
        if len(pattern) >= 2:
            # For pattern [A, B, C, ...], the rule is A,B,... → last_item
            antecedent = pattern[:-1]
            consequent = pattern[-1]
            
            # Find support count of antecedent
            antecedent_count = None
            if len(antecedent) == 1:
                antecedent_count = single_item_support.get(antecedent[0])
            else:
                # Find antecedent in patterns
                for c, p in patterns:
                    if p == list(antecedent):
                        antecedent_count = c
                        break
            
            if antecedent_count:
                support = count / total_sequences
                antecedent_support = antecedent_count / total_sequences
                confidence = count / antecedent_count
                
                rules.append({
                    'antecedent': antecedent,
                    'consequent': consequent,
                    'support': support,
                    'antecedent_support': antecedent_support,
                    'confidence': confidence,
                    'count': count,
                    'antecedent_count': antecedent_count
                })
    
    return rules

# Set minimum confidence threshold
min_conf = 0.5  # Example: only show rules with confidence >= 50%

# Get rules with confidence
rules = calculate_confidence(patterns, total_sequences)

# Filter rules by minimum confidence
rules = [rule for rule in rules if rule['confidence'] >= min_conf]

# Sort by confidence (descending), then support (descending)
rules_sorted = sorted(rules, key=lambda x: (-x['confidence'], -x['support']))

# Display sequential rules with confidence
print(f"Total sequences: {total_sequences}")
print(f"Found {len(rules_sorted)} sequential rules (min support = {min_support}/{total_sequences}, min confidence = {min_conf:.0%}):\n")
print("="*100)

for i, rule in enumerate(rules_sorted, 1):
    print(f"\nRule {i}:")
    print(f"Support: {rule['support']:.2%} ({rule['count']}/{total_sequences}) | Confidence: {rule['confidence']:.2%} ({rule['count']}/{rule['antecedent_count']})")
    print(f"\nIF borrowed:")
    for j, book in enumerate(rule['antecedent'], 1):
        print(f"  {j}. {book}")
    print(f"\nTHEN will borrow:")
    print(f"  → {rule['consequent']}")
    print("-"*100)

Total sequences: 119
Found 81 sequential rules (min support = 1/119, min confidence = 50%):


Rule 1:
Support: 1.68% (2/119) | Confidence: 100.00% (2/2)

IF borrowed:
  1. NEURAL NETWORKS AND DEEP LEARNING

THEN will borrow:
  → COMPUTER VISION ALGORITHMS AND APPLICATIONS : ALGORITHMS AND APPLICATIONS
----------------------------------------------------------------------------------------------------

Rule 2:
Support: 0.84% (1/119) | Confidence: 100.00% (1/1)

IF borrowed:
  1. RÉSEAUX INFORMATIQUES : RECUEIL DE SUJETS D'EXAMENS AVEC SOLUTIONS

THEN will borrow:
  → PROGRAMMATION LINÉAIRE
----------------------------------------------------------------------------------------------------

Rule 3:
Support: 0.84% (1/119) | Confidence: 100.00% (1/1)

IF borrowed:
  1. RÉSEAUX INFORMATIQUES : RECUEIL DE SUJETS D'EXAMENS AVEC SOLUTIONS
  2. PROGRAMMATION LINÉAIRE

THEN will borrow:
  → RECHERCHE OPÉRATIONNELLE POUR INGÉNIEURS. 2
--------------------------------------------------------------

In [4]:
# evaluation
# evaluation

def evaluate_rules_with_metrics(rules, total_sequences):
    """
    Adds lift, leverage, and conviction to each rule.
    """
    for rule in rules:
        support = rule['support']
        antecedent_support = rule['antecedent_support']
        consequent_support = None

        # Calculate consequent support
        # Consequent is a single book title
        consequent_count = 0
        for r in rules:
            if r['antecedent'] == [rule['consequent']]:
                consequent_count = r['count']
                break
        if consequent_count == 0:
            # Fallback: count how many sequences contain the consequent
            consequent_count = sum([rule['consequent'] in seq for seq in sequences])
        consequent_support = consequent_count / total_sequences

        # Calculate metrics
        lift = rule['confidence'] / consequent_support if consequent_support > 0 else 0
        leverage = support - (antecedent_support * consequent_support)
        conviction = (1 - consequent_support) / (1 - rule['confidence']) if rule['confidence'] < 1 else float('inf')

        rule['lift'] = lift
        rule['leverage'] = leverage
        rule['conviction'] = conviction

    return rules

# Evaluate rules
rules_evaluated = evaluate_rules_with_metrics(rules_sorted, total_sequences)

# Display rules with new metrics
for i, rule in enumerate(rules_evaluated, 1):
    print(f"\nRule {i}:")
    print(f"Support: {rule['support']:.2%} | Confidence: {rule['confidence']:.2%} | Lift: {rule['lift']:.2f} | Leverage: {rule['leverage']:.4f} | Conviction: {rule['conviction']:.2f}")
    print(f"IF borrowed: {', '.join(rule['antecedent'])}")
    print(f"THEN will borrow: {rule['consequent']}")
    print("-"*100)



Rule 1:
Support: 1.68% | Confidence: 100.00% | Lift: 119.00 | Leverage: 0.0167 | Conviction: inf
IF borrowed: NEURAL NETWORKS AND DEEP LEARNING
THEN will borrow: COMPUTER VISION ALGORITHMS AND APPLICATIONS : ALGORITHMS AND APPLICATIONS
----------------------------------------------------------------------------------------------------

Rule 2:
Support: 0.84% | Confidence: 100.00% | Lift: 119.00 | Leverage: 0.0083 | Conviction: inf
IF borrowed: RÉSEAUX INFORMATIQUES : RECUEIL DE SUJETS D'EXAMENS AVEC SOLUTIONS
THEN will borrow: PROGRAMMATION LINÉAIRE
----------------------------------------------------------------------------------------------------

Rule 3:
Support: 0.84% | Confidence: 100.00% | Lift: 119.00 | Leverage: 0.0083 | Conviction: inf
IF borrowed: RÉSEAUX INFORMATIQUES : RECUEIL DE SUJETS D'EXAMENS AVEC SOLUTIONS, PROGRAMMATION LINÉAIRE
THEN will borrow: RECHERCHE OPÉRATIONNELLE POUR INGÉNIEURS. 2
------------------------------------------------------------------------------